In [1]:
# 📘 HuBERT Transformer Training on Synthetic GAN Audio Data

import os
import torch
import torchaudio
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Processor, HubertModel
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
from tqdm import tqdm

In [2]:
# ----------------------------
# Config
# ----------------------------
class Config:
    SAMPLE_RATE = 16000
    MODEL_NAME = "facebook/hubert-base-ls960"
    BATCH_SIZE = 4
    EPOCHS = 50
    LR = 1e-3
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_PATH = Path("generated_samples_fan/final_samples")

config = Config()

# ----------------------------
# Dataset
# ----------------------------
class AudioDataset(Dataset):
    def __init__(self, data_path):
        self.filepaths = []
        self.labels = []

        for label in ["normal", "abnormal"]:
            folder = data_path / label
            for file in folder.glob("*.wav"):
                self.filepaths.append(file)
                self.labels.append(label)

        self.label_encoder = LabelEncoder()
        self.encoded_labels = self.label_encoder.fit_transform(self.labels)

        # self.processor = Wav2Vec2Processor.from_pretrained(config.MODEL_NAME)

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        waveform, sr = torchaudio.load(path)
        waveform = torchaudio.functional.resample(waveform, sr, config.SAMPLE_RATE)
        # input_values = self.processor(waveform.squeeze().numpy(), sampling_rate=config.SAMPLE_RATE, return_tensors="pt").input_values.squeeze(0)
        input_values = waveform.squeeze(0)  # Use raw waveform for HuBERT
        label = torch.tensor(self.encoded_labels[idx], dtype=torch.long)
        return input_values, label

In [ ]:
# ----------------------------
# Transformer Model
# ----------------------------
class HubertClassifier(nn.Module):
    def __init__(self, hidden_dim=768, num_classes=2):
        super(HubertClassifier, self).__init__()
        self.hubert = HubertModel.from_pretrained(config.MODEL_NAME)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_values):
        with torch.no_grad():
            features = self.hubert(input_values).last_hidden_state
        pooled = torch.mean(features, dim=1)
        logits = self.classifier(pooled)
        return logits

# ----------------------------
# Training Loop
# ----------------------------
def train():
    dataset = AudioDataset(config.DATA_PATH)
    dataloader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True)

    model = HubertClassifier().to(config.DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.classifier.parameters(), lr=config.LR)

    model.train()
    for epoch in range(config.EPOCHS):
        running_loss = 0.0
        for inputs, labels in tqdm(dataloader):
            inputs = inputs.to(config.DEVICE)
            labels = labels.to(config.DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch + 1}/{config.EPOCHS}, Loss: {running_loss:.4f}")

    torch.save(model.state_dict(), "hubert_transformer_synthetic.pth")

# ----------------------------
# Run
# ----------------------------
if __name__ == '__main__':
    train()

c:\Users\Qinmay\anaconda3\envs\py310_env\lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\Qinmay\anaconda3\envs\py310_env\lib\site-packages\transformers\modeling_utils.py:442: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowl

Epoch 1/10, Loss: 69.7078


100%|██████████| 100/100 [00:59<00:00,  1.68it/s]


Epoch 2/10, Loss: 69.6965


100%|██████████| 100/100 [00:48<00:00,  2.05it/s]


...

Epoch 8/10, Loss: 69.3982


100%|██████████| 100/100 [00:16<00:00,  6.05it/s]


Epoch 9/10, Loss: 69.9196


100%|██████████| 100/100 [00:29<00:00,  3.42it/s]


Epoch 10/10, Loss: 69.6556
